# T31-机构行为监测 - 数据获取模块

## 1. 模块概述

数据获取模块负责从数据库和文件系统中获取机构行为监测所需的各类数据。主要包括：

- 机构交易统计数据获取
- 交易明细数据获取
- 债券基本信息获取
- 市场利率数据获取
- Excel文件数据导入

## 2. 数据库连接管理

### 2.1 配置管理

使用环境变量管理数据库连接信息，避免硬编码密码。

In [ ]:
import os
import sys
import datetime
import pandas as pd
import numpy as np
from typing import Optional, List, Dict, Any

# 配置类 - 使用环境变量
class DatabaseConfig:
    """数据库配置类"""
    
    def __init__(self):
        self.user = os.getenv('DB_USER', 'hz_work')
        self.password = os.getenv('DB_PASSWORD', '')
        self.host = os.getenv('DB_HOST', 'rm-uf6c8yi6zdq6ea7p1qo.mysql.rds.aliyuncs.com')
        self.port = os.getenv('DB_PORT', '3306')
        self.database = os.getenv('DB_NAME', 'bond')
        self.charset = 'utf8'
    
    def get_connection_url(self) -> str:
        """获取SQLAlchemy连接URL"""
        return f"mysql+pymysql://{self.user}:{self.password}@{self.host}:{self.port}/{self.database}?charset={self.charset}"

# 创建配置实例
config = DatabaseConfig()
print(f"数据库配置: {config.host}:{config.port}/{config.database}")

In [ ]:
# 数据库管理器
from sqlalchemy import create_engine, text
from sqlalchemy.pool import NullPool
import logging

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

class DatabaseManager:
    """数据库管理器"""
    
    def __init__(self, config: DatabaseConfig):
        self.config = config
        self.engine = None
    
    def connect(self):
        """建立数据库连接"""
        if self.engine is None:
            try:
                connection_url = self.config.get_connection_url()
                self.engine = create_engine(
                    connection_url,
                    poolclass=NullPool,
                    echo=False,
                    pool_recycle=3600
                )
                logger.info("数据库连接已建立")
            except Exception as e:
                logger.error(f"数据库连接失败: {e}")
                raise
        return self.engine
    
    def execute_query(self, sql: str) -> pd.DataFrame:
        """执行查询并返回DataFrame"""
        try:
            engine = self.connect()
            df = pd.read_sql(sql, engine)
            logger.info(f"查询执行成功，返回 {len(df)} 行数据")
            return df
        except Exception as e:
            logger.error(f"查询执行失败: {e}")
            raise
    
    def test_connection(self) -> bool:
        """测试数据库连接"""
        try:
            engine = self.connect()
            with engine.connect() as conn:
                conn.execute(text("SELECT 1"))
            logger.info("数据库连接测试成功")
            return True
        except Exception as e:
            logger.error(f"数据库连接测试失败: {e}")
            return False
    
    def close(self):
        """关闭数据库连接"""
        if self.engine:
            self.engine.dispose()
            self.engine = None
            logger.info("数据库连接已关闭")

# 创建数据库管理器实例
db_manager = DatabaseManager(config)
print("数据库管理器已创建")

## 3. 机构交易统计数据获取

### 3.1 现券成交分机构统计表

这是项目核心数据表，包含各机构类型按期限、券种分类的交易统计。

In [ ]:
class InstitutionDataFetcher:
    """机构交易数据获取器"""
    
    def __init__(self, db_manager: DatabaseManager):
        self.db_manager = db_manager
    
    def get_daily_net_inflow(self, trade_date: str) -> pd.DataFrame:
        """
        获取各机构类型每日净交易量
        
        Args:
            trade_date: 交易日期 (YYYY-MM-DD)
        
        Returns:
            DataFrame: 包含机构类型和净买入交易量
        """
        query = f"""
        SELECT
            `机构类型`,
            `净买入交易量（亿元）`
        FROM
            bond.`现券成交分机构统计表`
        WHERE
            `交易日期` = '{trade_date}';
        """
        df = self.db_manager.execute_query(query)
        
        # 数据清洗
        df['净买入交易量（亿元）'] = df['净买入交易量（亿元）'].replace('--', '0')
        df['净买入交易量（亿元）'] = pd.to_numeric(df['净买入交易量（亿元）'])
        
        # 按机构类型聚合
        df_chart_data = df.groupby('机构类型')['净买入交易量（亿元）'].sum().reset_index()
        df_chart_data.rename(columns={'机构类型': 'name', '净买入交易量（亿元）': 'value'}, inplace=True)
        
        return df_chart_data
    
    def get_tenor_preference(self, trade_date: str, institution_type: str) -> pd.DataFrame:
        """
        获取机构期限偏好数据
        
        Args:
            trade_date: 交易日期
            institution_type: 机构类型
        
        Returns:
            DataFrame: 期限偏好数据
        """
        query = f"""
        SELECT 
            `期限`, 
            `买入交易量（亿元）` as buy_vol, 
            `卖出交易量（亿元）` as sell_vol
        FROM bond.`现券成交分机构统计表`
        WHERE `交易日期` = '{trade_date}' AND `机构类型` = '{institution_type}';
        """
        df = self.db_manager.execute_query(query)
        
        # 数据清洗
        df['buy_vol'] = pd.to_numeric(df['buy_vol'].replace('--', '0'))
        df['sell_vol'] = pd.to_numeric(df['sell_vol'].replace('--', '0'))
        
        # 计算总交易量
        df['total_vol'] = df['buy_vol'] + df['sell_vol']
        
        # 过滤无交易期限
        df = df[df['total_vol'] > 0]
        
        # 计算占比
        total = df['total_vol'].sum()
        if total > 0:
            df['percentage'] = (df['total_vol'] / total) * 100
        else:
            df['percentage'] = 0
        
        df.rename(columns={'期限': 'name', 'total_vol': 'value'}, inplace=True)
        
        return df
    
    def get_bond_type_preference(self, trade_date: str, institution_type: str) -> pd.DataFrame:
        """
        获取机构券种偏好数据
        
        Args:
            trade_date: 交易日期
            institution_type: 机构类型
        
        Returns:
            DataFrame: 券种偏好数据
        """
        query = f"""
        SELECT 
            `债券类型`, 
            `买入交易量（亿元）` as buy_vol, 
            `卖出交易量（亿元）` as sell_vol
        FROM bond.`现券成交分机构统计表`
        WHERE `交易日期` = '{trade_date}' AND `机构类型` = '{institution_type}';
        """
        df = self.db_manager.execute_query(query)
        
        # 数据清洗
        df['buy_vol'] = pd.to_numeric(df['buy_vol'].replace('--', '0'))
        df['sell_vol'] = pd.to_numeric(df['sell_vol'].replace('--', '0'))
        df['total_vol'] = df['buy_vol'] + df['sell_vol']
        df = df[df['total_vol'] > 0]
        
        total = df['total_vol'].sum()
        if total > 0:
            df['percentage'] = (df['total_vol'] / total) * 100
        else:
            df['percentage'] = 0
        
        df.rename(columns={'债券类型': 'name', 'total_vol': 'value'}, inplace=True)
        
        return df
    
    def get_market_share_time_series(self, start_date: str, end_date: str) -> pd.DataFrame:
        """
        获取市场份额时序数据
        
        Args:
            start_date: 开始日期
            end_date: 结束日期
        
        Returns:
            DataFrame: 市场份额时序数据
        """
        query = f"""
        SELECT 
            `交易日期`, 
            `机构类型`, 
            `买入交易量（亿元）` as buy_vol, 
            `卖出交易量（亿元）` as sell_vol
        FROM bond.`现券成交分机构统计表`
        WHERE `交易日期` BETWEEN '{start_date}' AND '{end_date}';
        """
        df = self.db_manager.execute_query(query)
        
        # 数据清洗
        df['buy_vol'] = pd.to_numeric(df['buy_vol'].replace('--', '0'))
        df['sell_vol'] = pd.to_numeric(df['sell_vol'].replace('--', '0'))
        df['total_vol'] = df['buy_vol'] + df['sell_vol']
        
        # 计算每日市场总量
        daily_total = df.groupby('交易日期')['total_vol'].sum().reset_index()
        daily_total.rename(columns={'total_vol': 'market_total_vol'}, inplace=True)
        
        # 计算机构日度总量
        daily_inst = df.groupby(['交易日期', '机构类型'])['total_vol'].sum().reset_index()
        
        # 合并计算市场份额
        df_final = pd.merge(daily_inst, daily_total, on='交易日期')
        df_final['market_share_percent'] = (df_final['total_vol'] / df_final['market_total_vol']) * 100
        
        # 透视为宽表
        df_chart_data = df_final.pivot(index='交易日期', columns='机构类型', values='market_share_percent').fillna(0)
        
        return df_chart_data

# 创建数据获取器
fetcher = InstitutionDataFetcher(db_manager)
print("机构数据获取器已创建")

## 4. 交易明细数据获取

### 4.1 dealtinfo表查询

交易明细数据用于深度分析机构行为。

In [ ]:
class TradeDetailFetcher:
    """交易明细数据获取器"""
    
    def __init__(self, db_manager: DatabaseManager):
        self.db_manager = db_manager
    
    def get_dealtinfo_with_bond_type(self, trade_date: str) -> pd.DataFrame:
        """
        获取交易明细并关联债券类型
        
        Args:
            trade_date: 交易日期
        
        Returns:
            DataFrame: 关联债券类型的交易明细
        """
        query = f"""
        SELECT 
            d.TRADE_CODE, 
            COALESCE(bc.ths_ths_bond_third_type_bond, bf.ths_ths_bond_third_type_bond, br.ths_ths_bond_third_type_bond) AS bond_type,
            d.transaction_amount/10000 as transaction_amount, 
            d.remaining_term,
            d.weighted_YTM
        FROM bond.dealtinfo d
        LEFT JOIN bond.basicinfo_credit bc ON d.TRADE_CODE = bc.trade_code
        LEFT JOIN bond.basicinfo_finance bf ON d.TRADE_CODE = bf.trade_code  
        LEFT JOIN bond.basicinfo_rate br ON d.TRADE_CODE = br.trade_code
        WHERE d.DT = '{trade_date}'
            AND (bc.trade_code IS NOT NULL OR bf.trade_code IS NOT NULL OR br.trade_code IS NOT NULL);
        """
        return self.db_manager.execute_query(query)
    
    def get_top_active_bonds(self, start_date: str, end_date: str, top_n: int = 10) -> pd.DataFrame:
        """
        获取十大活跃券
        
        Args:
            start_date: 开始日期
            end_date: 结束日期
            top_n: 返回前N只
        
        Returns:
            DataFrame: 活跃券统计
        """
        query = f"""
        SELECT 
            d.TRADE_CODE, 
            COALESCE(bc.SEC_NAME, bf.SEC_NAME, br.SEC_NAME) AS SEC_NAME,
            d.transaction_amount/10000 as transaction_amount, 
            d.DT
        FROM bond.dealtinfo d
        LEFT JOIN bond.basicinfo_credit bc ON d.TRADE_CODE = bc.trade_code
        LEFT JOIN bond.basicinfo_finance bf ON d.TRADE_CODE = bf.trade_code  
        LEFT JOIN bond.basicinfo_rate br ON d.TRADE_CODE = br.trade_code
        WHERE d.DT BETWEEN '{start_date}' AND '{end_date}'
            AND COALESCE(bc.SEC_NAME, bf.SEC_NAME, br.SEC_NAME) IS NOT NULL;
        """
        df = self.db_manager.execute_query(query)
        
        if df.empty:
            return pd.DataFrame()
        
        # 按债券汇总
        bond_stats = df.groupby('SEC_NAME').agg({
            'transaction_amount': 'sum',
            'TRADE_CODE': 'count'
        }).reset_index()
        
        bond_stats.rename(columns={
            'SEC_NAME': 'name',
            'transaction_amount': 'total_amount',
            'TRADE_CODE': 'trade_count'
        }, inplace=True)
        
        # 排序取前N
        df_top = bond_stats.sort_values(by='total_amount', ascending=False).head(top_n)
        df_top['type'] = f'成交金额TOP{top_n}'
        df_top['value'] = df_top['total_amount']
        
        return df_top

# 创建交易明细获取器
trade_fetcher = TradeDetailFetcher(db_manager)
print("交易明细获取器已创建")

## 5. 城投债数据获取

### 5.1 城投债利差数据

In [ ]:
class CityInvestBondFetcher:
    """城投债数据获取器"""
    
    def __init__(self, db_manager: DatabaseManager):
        self.db_manager = db_manager
    
    def get_city_bond_spread(self, trade_date: str) -> pd.DataFrame:
        """
        获取城投债利差数据
        
        Args:
            trade_date: 交易日期
        
        Returns:
            DataFrame: 各省份城投债利差
        """
        # 获取城投债交易数据
        city_bond_query = f"""
        SELECT 
            d.DT, 
            d.weighted_YTM, 
            b.ths_issuer_respond_district_bond_province AS province,
            b.ths_bond_maturity_theory_bond AS tenor
        FROM 
            bond.dealtinfo d
        JOIN 
            bond.basicinfo_credit b ON d.TRADE_CODE = b.trade_code
        WHERE 
            d.DT = '{trade_date}' AND b.ths_is_city_invest_debt_yy_bond = '是';
        """
        df_city_bond = self.db_manager.execute_query(city_bond_query)
        
        # 获取国债收益率基准
        rate_bond_query = f"""
        SELECT 
            tenor, 
            AVG(weighted_YTM) as risk_free_rate
        FROM (
            SELECT 
                d.weighted_YTM, 
                b.ths_bond_maturity_theory_bond AS tenor
            FROM 
                bond.dealtinfo d
            JOIN 
                bond.basicinfo_rate b ON d.TRADE_CODE = b.trade_code
            WHERE 
                d.DT = '{trade_date}'
        ) as subquery
        GROUP BY tenor;
        """
        df_rate_bond = self.db_manager.execute_query(rate_bond_query)
        
        # 合并计算利差
        df_merged = pd.merge(df_city_bond, df_rate_bond, on='tenor', how='left')
        df_merged.dropna(subset=['weighted_YTM', 'risk_free_rate'], inplace=True)
        df_merged['spread'] = (df_merged['weighted_YTM'] - df_merged['risk_free_rate']) * 100  # BP
        
        # 按省份聚合
        df_chart_data = df_merged.groupby('province')['spread'].mean().reset_index()
        df_chart_data.rename(columns={'province': 'name', 'spread': 'value'}, inplace=True)
        
        return df_chart_data
    
    def get_city_bond_fund_source(self, province: str, start_date: str, end_date: str) -> pd.DataFrame:
        """
        获取区域城投债资金来源
        
        Args:
            province: 省份名称
            start_date: 开始日期
            end_date: 结束日期
        
        Returns:
            DataFrame: 资金来源统计
        """
        # 获取省份城投债列表
        city_bond_list_query = f"""
        SELECT DISTINCT `债券简称` as sec_name 
        FROM bond.basicinfo_credit 
        WHERE ths_is_city_invest_debt_yy_bond = '是' 
        AND ths_issuer_respond_district_bond_province = '{province}';
        """
        df_city_bonds = self.db_manager.execute_query(city_bond_list_query)
        
        if df_city_bonds.empty:
            return pd.DataFrame(columns=['source', 'value', 'target'])
        
        city_bond_list = tuple(df_city_bonds['sec_name'].tolist())
        
        # 查询资金流向
        if len(city_bond_list) == 1:
            bond_condition = f"= '{city_bond_list[0]}'"
        else:
            bond_condition = f"IN {city_bond_list}"
        
        flow_query = f"""
        SELECT 
            `机构类型` as source,
            SUM(`买入交易量（亿元）`) as value
        FROM 
            bond.`现券成交分机构统计表` 
        WHERE 
            `交易日期` BETWEEN '{start_date}' AND '{end_date}'
            AND `债券简称` {bond_condition}
        GROUP BY 
            `机构类型`;
        """
        df_chart_data = self.db_manager.execute_query(flow_query)
        
        if df_chart_data.empty:
            return pd.DataFrame(columns=['source', 'value', 'target'])
        
        df_chart_data.dropna(inplace=True)
        df_chart_data = df_chart_data[df_chart_data['value'] > 0]
        df_chart_data['target'] = province
        
        return df_chart_data

# 创建城投债获取器
city_bond_fetcher = CityInvestBondFetcher(db_manager)
print("城投债数据获取器已创建")

## 6. 资金面数据获取

### 6.1 回购余额和利率数据

In [ ]:
class FundConditionFetcher:
    """资金面数据获取器"""
    
    def __init__(self, db_manager: DatabaseManager):
        self.db_manager = db_manager
    
    def get_repo_data(self, start_date: str, end_date: str) -> Dict[str, Any]:
        """
        获取回购余额和利率数据
        
        Args:
            start_date: 开始日期
            end_date: 结束日期
        
        Returns:
            Dict: 包含余额和利率数据
        """
        # 银行间回购余额
        repo_balance_interbank_query = f"""
        SELECT
            A.dt,
            sum(A.close) as interbank_repo_balance
        FROM edb.edbdata A
        WHERE A.dt >= '{start_date}'
          AND A.dt <= '{end_date}'
          AND A.trade_code ='M0041739'
        GROUP BY A.dt
        ORDER BY dt
        """
        df_interbank = self.db_manager.execute_query(repo_balance_interbank_query)
        
        # 交易所回购余额
        repo_balance_exchange_query = f"""
        SELECT
            A.dt,
            sum(A.close*1000/100000000) as exchange_repo_balance
        FROM edb.edbdata A
        WHERE A.dt >= '{start_date}'
          AND A.dt <= '{end_date}'
          AND A.trade_code in ('M0340576','M0340580','M0340581')
        GROUP BY A.dt
        ORDER BY dt
        """
        df_exchange = self.db_manager.execute_query(repo_balance_exchange_query)
        
        # 回购利率 (DR007, R007)
        repo_rate_query = f"""
        SELECT A.DT, A.CLOSE AS DR007, B.CLOSE AS R007 FROM 
          (SELECT * FROM bond.marketinfo_curve WHERE TRADE_CODE='L001619493' AND DT>='{start_date}') A 
        LEFT JOIN
          (SELECT * FROM bond.marketinfo_curve WHERE TRADE_CODE='L004369617' AND DT>='{start_date}') B 
        ON A.DT=B.DT
        WHERE A.DT <= '{end_date}'
        """
        df_rates = self.db_manager.execute_query(repo_rate_query)
        
        # 合并数据
        df_balances = pd.merge(df_interbank, df_exchange, on='dt', how='outer')
        df_balances.sort_values(by='dt', inplace=True)
        df_balances.ffill(inplace=True)
        df_balances['total_repo_balance'] = df_balances['interbank_repo_balance'].fillna(0) + df_balances['exchange_repo_balance'].fillna(0)
        
        df_final = pd.merge(df_balances, df_rates, left_on='dt', right_on='DT', how='outer')
        df_final.sort_values(by='dt', inplace=True)
        df_final.ffill(inplace=True)
        
        # 返回最新一天数据
        if not df_final.empty:
            latest = df_final.tail(1).iloc[0]
            return {
                'date': latest['dt'],
                'total_repo_balance': latest['total_repo_balance'],
                'dr007': latest['DR007'],
                'r007': latest['R007']
            }
        
        return {}

# 创建资金面获取器
fund_fetcher = FundConditionFetcher(db_manager)
print("资金面数据获取器已创建")

## 7. Excel文件数据导入

### 7.1 数据同步工具

In [ ]:
class ExcelDataImporter:
    """Excel数据导入器"""
    
    def __init__(self, db_manager: DatabaseManager):
        self.db_manager = db_manager
    
    def find_header_row(self, file_path: str) -> int:
        """
        智能查找Excel文件中的标题行
        
        Args:
            file_path: Excel文件路径
        
        Returns:
            int: 标题行索引
        """
        try:
            df_peek = pd.read_excel(file_path, header=None, nrows=10)
            for i, row in df_peek.iterrows():
                if any(isinstance(cell, str) and ('日期' in cell or '机构' in cell) for cell in row.values):
                    return i
        except Exception:
            pass
        return 0
    
    def load_excel_data(self, file_path: str) -> pd.DataFrame:
        """
        加载Excel数据
        
        Args:
            file_path: Excel文件路径
        
        Returns:
            DataFrame: 加载的数据
        """
        import os
        
        if not os.path.exists(file_path):
            raise FileNotFoundError(f"文件不存在: {file_path}")
        
        # 智能查找标题行
        header_row = self.find_header_row(file_path)
        df = pd.read_excel(file_path, header=header_row)
        
        # 清理空列
        df.dropna(axis=1, how='all', inplace=True)
        
        print(f"成功从 {os.path.basename(file_path)} 加载 {len(df)} 条记录")
        
        return df
    
    def clean_volume_data(self, df: pd.DataFrame) -> pd.DataFrame:
        """
        清洗交易量数据
        
        Args:
            df: 原始DataFrame
        
        Returns:
            DataFrame: 清洗后的数据
        """
        df = df.copy()
        
        volume_columns = ['净买入交易量（亿元）', '买入交易量（亿元）', '卖出交易量（亿元）']
        
        for col in volume_columns:
            if col in df.columns:
                df[col] = df[col].replace('--', 0)
                df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)
        
        # 日期列处理
        if '交易日期' in df.columns:
            df['交易日期'] = pd.to_datetime(df['交易日期']).dt.date
        
        return df

# 创建Excel导入器
excel_importer = ExcelDataImporter(db_manager)
print("Excel数据导入器已创建")

## 8. 使用示例

In [ ]:
# 使用示例
def demo_data_fetching():
    """演示数据获取功能"""
    
    # 设置日期
    today = datetime.datetime.now().strftime('%Y-%m-%d')
    start_date = (datetime.datetime.now() - datetime.timedelta(days=30)).strftime('%Y-%m-%d')
    
    print("=" * 60)
    print("机构行为监测 - 数据获取示例")
    print("=" * 60)
    
    # 1. 测试数据库连接
    print("\n1. 测试数据库连接...")
    if db_manager.test_connection():
        print("   数据库连接成功")
    else:
        print("   数据库连接失败")
    
    # 2. 获取机构类型列表
    print("\n2. 机构类型列表:")
    institution_types = [
        '大型商业银行/政策性银行',
        '股份制商业银行',
        '城市商业银行',
        '农村金融机构',
        '证券公司',
        '基金公司及产品',
        '保险公司',
        '境外机构'
    ]
    for inst in institution_types:
        print(f"   - {inst}")
    
    print("\n" + "=" * 60)
    print("数据获取模块准备就绪")
    print("=" * 60)

# 执行演示
demo_data_fetching()

In [ ]:
# 清理资源
def cleanup():
    """清理数据库连接"""
    db_manager.close()
    print("数据库连接已关闭")

# 注册清理函数
import atexit
atexit.register(cleanup)

print("模块加载完成")